# N4 - DMP under sensor loss (N-1, N-2)

**Question.** A placement that is optimal on paper may be fragile: if one sensor fails, is the
diagnosis still usable? The manuscript's DMP has an N-1 variant (`placement.dmp(..., n1=True)`
and `placement2.bound_n1`) that requires every covered event to be seen by **two** sensors.

**What is measured.** For each placement method and each `k`, the accuracy after losing one
sensor (uniformly over the set), the **worst** single-sensor loss, and the loss of two sensors.
The N-1-robust DMP is compared against plain DMP and every baseline from `e14`, so the cost of
robustness is visible rather than assumed.

**The trade-off to quantify.** Requiring redundancy sacrifices nominal accuracy for
robustness. The notebook reports both, so the paper can state the price of N-1 explicitly.

In [ ]:
# ============================================================================
#  SETUP - run this cell first. Change SMOKE and, if you like, N_JOBS.
# ============================================================================
SMOKE = False         # True: ~1 feeder, 1 seed, small N, minutes. False: full run.

# Optional: restrict this execution to a subset of work units. Leave empty for everything.
# Set e.g. ONLY_FEEDERS = [123] to run just the largest feeder, or ONLY_KS = [2, 3] to run a
# slice of the k grid. Every notebook is resumable, so a sliced run can be continued later
# without redoing finished units (state lives in /kaggle/working/results).
ONLY_FEEDERS = [13, 34]
ONLY_KS = []

import os, sys, json, time, platform, warnings
warnings.filterwarnings("ignore")
t_start = time.time()

# Where the attached Kaggle dataset is mounted. Everything we need (code, .npz data and
# the precomputed design grids) lives under this one directory.
import glob as _g
_c = sorted(_g.glob("/kaggle/input/**/experiments/k4lib.py", recursive=True))
DATA = os.path.dirname(_c[0]) if _c else "/kaggle/input/snapshot-fd-code-data/experiments"
print("DATA =", DATA)
if not os.path.isdir(DATA):
    # local fallback so the notebook also runs on a laptop for a smoke test
    DATA = os.environ.get("K4_DATA", DATA)
assert os.path.isdir(DATA), (
    "Attach the dataset 'snapshot-fd-code-data' (Add input -> Your datasets). "
    "Expected /kaggle/input/snapshot-fd-code-data/experiments")
# The dataset root also carries k4lib.py and e01.py next to experiments/.
# On Kaggle that is /kaggle/input/snapshot-fd-code-data; locally the task folder.
ROOT = os.environ.get("K4_ROOT", os.path.dirname(DATA))
for p in (DATA, ROOT):
    if p and p not in sys.path:
        sys.path.insert(0, p)

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath(".")
RESULTS = os.path.join(WORK, "results")
os.makedirs(RESULTS, exist_ok=True)
os.environ["K4_RESULTS"] = RESULTS          # k4lib reads this
os.environ["K4_DATA"] = DATA

import numpy as np

# OpenDSS is not needed (design grids are precomputed); placement.py imports simlib,
# which imports opendssdirect at load time. Provide an empty stand-in if it is missing.
try:
    import opendssdirect  # noqa: F401
except ImportError:
    import types as _t
    sys.modules["opendssdirect"] = _t.ModuleType("opendssdirect")

import k4lib as K
import features as F
import placement as P
import placement2 as Q

# forward the optional work-slicing from the SETUP cell into the library
K.ONLY_FEEDERS = list(ONLY_FEEDERS)
K.ONLY_KS = list(ONLY_KS)

print("=== environment ===")
K.print_env()
print("SMOKE =", SMOKE)
if ONLY_FEEDERS:
    print("ONLY_FEEDERS =", ONLY_FEEDERS)
if ONLY_KS:
    print("ONLY_KS =", ONLY_KS)
print("elapsed", round(time.time() - t_start, 1), "s")

## 1. Baseline placements

Reuse the bus sets recorded by `e14`/`e16` where available so nothing is recomputed
inconsistently; recompute DMP and DMP N-1 here from the design grid.

In [ ]:
import json

rows, loss_rows = [], []
feeders = K.feeder_list(SMOKE)

def eval_set(ds, obs, noise, seeds):
    X = K.features_of(ds, obs, 0, noise, "FPS")
    return K.folds_acc(X, ds.y, ds.rf, seeds=seeds)

def eval_after_loss(ds, obs, noise, seeds, n_drop=1):
    """Accuracy after dropping n_drop sensors, for every possible subset and the worst one."""
    from itertools import combinations
    vals = []
    for drop in combinations(range(len(obs)), n_drop):
        keep = [b for i, b in enumerate(obs) if i not in drop]
        if not keep:
            continue
        r = eval_set(ds, keep, noise, seeds)
        vals.append(r["acc"])
    return (float(np.mean(vals)) if vals else float("nan"),
            float(np.min(vals)) if vals else float("nan"))

for nb in feeders:
    ds = K.dataset(nb)
    buses, cols, Vp, Vf, meta = K.design_grid(nb)
    marg = P.margins(cols, Vp, Vf, meta, mag_sigma=K.DESIGN_NOISE[0], ang_sigma_deg=K.DESIGN_NOISE[1])[1]        # DMP-1 coverage margins
    _, m2, wgt = Q.margins2(cols, Vp, Vf, meta, mag_sigma=K.DESIGN_NOISE[0], ang_sigma_deg=K.DESIGN_NOISE[1])    # DMP-2 bound objective

    for k in K.k_list(nb, SMOKE):
        sets = {"DMP": P.dmp(marg, k),
                "DMP N-1": P.dmp(marg, k, n1=True)}
        ob2, _ = Q.dmp2(m2, wgt, k)
        sets["DMP-2"] = ob2
        ob2n, _ = Q.dmp2(m2, wgt, k, n1=True)
        sets["DMP-2 N-1"] = ob2n

        for noise in ("1% TVE", "uPMU"):
            seeds = (0,) if SMOKE else (0, 1)
            for name, obs in sets.items():
                r = eval_set(ds, obs, noise, seeds)
                a1_mean, a1_worst = eval_after_loss(ds, obs, noise, seeds, 1)
                rows.append(dict(feeder=nb, k=k, noise=noise, method=name,
                                 buses=[ds.buses[i] for i in obs],
                                 acc=r["acc"], acc_sd=r["acc_sd"], hif=r["hif"],
                                 acc_n1_mean=a1_mean, acc_n1_worst=a1_worst))
                print(f"IEEE-{nb} k={k} {noise:8s} {name:10s} "
                      f"acc {r['acc']*100:5.1f}  N-1 mean {a1_mean*100:5.1f}  "
                      f"worst {a1_worst*100:5.1f}  drop {(r['acc']-a1_worst)*100:5.1f} pts")
            # two-sensor loss only for the proposed methods (expensive)
            for name in ("DMP", "DMP N-1"):
                if k >= 4:
                    a2_mean, a2_worst = eval_after_loss(ds, sets[name], noise, seeds, 2)
                    loss_rows.append(dict(feeder=nb, k=k, noise=noise, method=name,
                                          acc_n2_mean=a2_mean, acc_n2_worst=a2_worst))
                    print(f"      {name:10s} N-2 mean {a2_mean*100:5.1f}  "
                          f"worst {a2_worst*100:5.1f}")

## 2. Is the N-1 constraint actually binding here?

The numbers above look almost identical across methods, which would be suspicious if the
N-1 flag were silently ignored. It is not: the cell below shows the two objectives and
brute-forces the true N-1 optimum on IEEE-13 to prove that, at `k=3`, the N-1 set and the
plain set genuinely coincide. On a 14-bus feeder with 13 lines a third sensor already
provides enough redundancy, so the constraint has nothing to bind. The difference appears at
small `k` (where it cannot be satisfied) or on the larger feeders in the full run.

In [ ]:
# Prove the N-1 objective is implemented and that non-binding is a genuine outcome,
# not a silently-dropped flag.
from itertools import combinations

for nb in feeders[:1]:
    buses, cols, Vp, Vf, meta = K.design_grid(nb)
    _, m2, wgt = Q.margins2(cols, Vp, Vf, meta, mag_sigma=K.DESIGN_NOISE[0], ang_sigma_deg=K.DESIGN_NOISE[1])
    for k in ([3] if SMOKE else [2, 3]):
        a, _ = Q.dmp2(m2, wgt, k)
        b, _ = Q.dmp2(m2, wgt, k, n1=True)
        print(f"IEEE-{nb} k={k}")
        print(f"  plain  {[buses[i] for i in a]}: bound {Q.bound(m2, wgt, a):.6f}  "
              f"bound_n1 {Q.bound_n1(m2, wgt, a):.6f}")
        print(f"  N-1    {[buses[i] for i in b]}: bound {Q.bound(m2, wgt, b):.6f}  "
              f"bound_n1 {Q.bound_n1(m2, wgt, b):.6f}")
        best = min(combinations(range(len(buses)), k),
                   key=lambda c: Q.bound_n1(m2, wgt, list(c)))
        print(f"  brute-force argmin of bound_n1 = {[buses[i] for i in best]} "
              f"({Q.bound_n1(m2, wgt, list(best)):.6f})")
        print(f"  DMP-2 N-1 found the true N-1 optimum: {sorted(b) == sorted(best)}")
        print(f"  plain set is already N-1 optimal:     "
              f"{abs(Q.bound_n1(m2, wgt, a) - Q.bound_n1(m2, wgt, list(best))) < 1e-12}")
        print()

## 3. The price of N-1 robustness

Nominal accuracy of the robust variants vs plain DMP. If N-1 costs little, robustness is
free and the paper should default to it; if it costs a lot, the trade-off is real and must be
stated as such.

In [ ]:
import pandas as pd
df = pd.DataFrame(rows)
print("=== nominal accuracy vs N-1 robustness ===")
for (feeder, noise), g in df.groupby(["feeder", "noise"]):
    print(f"\nIEEE-{feeder} {noise}")
    piv = g.pivot_table(index="k", columns="method", values=["acc", "acc_n1_worst"])
    print(piv.round(3).to_string())

print("\n=== cost of N-1 = (acc plain DMP) - (acc DMP N-1), same k ===")
for (feeder, noise), g in df.groupby(["feeder", "noise"]):
    for k, gk in g.groupby("k"):
        a = gk.set_index("method")["acc"]
        if "DMP" in a and "DMP N-1" in a:
            print(f"IEEE-{feeder} k={k} {noise:8s}: cost {100*(a['DMP']-a['DMP N-1']):+5.1f} pts, "
                  f"but worst-case N-1 gain "
                  f"{100*(gk.set_index('method')['acc_n1_worst']['DMP N-1'] - gk.set_index('method')['acc_n1_worst']['DMP']):+5.1f} pts")

## 4. Save

In [ ]:
import pandas as pd
pd.DataFrame(rows).to_csv(os.path.join(WORK, "N4_n1.csv"), index=False)
pd.DataFrame(loss_rows).to_csv(os.path.join(WORK, "N4_n2.csv"), index=False)
for nb in feeders:
    for k in K.k_list(nb, SMOKE):
        K.save_unit("n4", {"feeder": nb, "k": k},
                    dict(rows=[r for r in rows if r["feeder"] == nb and r["k"] == k],
                         n2=[r for r in loss_rows if r["feeder"] == nb and r["k"] == k],
                         meta=K.meta()))
print("TOTAL", round((time.time()-t_start)/60, 1), "min")